# 🧹 Data Cleaning — Superstore Sales
**Author:** Senior Data Analyst | **Stack:** Pandas, NumPy

This notebook performs the full cleaning workflow: missing values, duplicates, outliers, validation, feature engineering, type conversion and standardization, then exports `Cleaned_Data.csv`.

> ℹ️ The raw file contains `Sales` only. We **derive** `Quantity`, `Discount` and `Profit` with seeded, reproducible business rules so the Profit/Margin/AOV analytics work end-to-end. These are clearly engineered fields.

In [ ]:
import numpy as np, pandas as pd
pd.set_option('display.max_columns', None)
SEED = 42
raw = pd.read_csv('../Dataset/Raw_Data.csv')
print(raw.shape)
raw.head()

## 1. Initial Profiling

In [ ]:
raw.info()

In [ ]:
raw.isnull().sum()[lambda s: s>0]

In [ ]:
print('Exact duplicate rows:', raw.duplicated().sum())

**Finding:** 9,800 rows × 18 cols. Only `Postal Code` has nulls (11). No exact duplicates.

## 2. Standardize Column Names

In [ ]:
df = raw.copy()
df.columns = (df.columns.str.strip().str.lower()
              .str.replace(' ', '_').str.replace('-', '_'))
list(df.columns)

## 3. Data Type Conversion
Dates are `dd/mm/yyyy`; cast dates, postal code and numerics.

In [ ]:
df['order_date'] = pd.to_datetime(df['order_date'], format='%d/%m/%Y', errors='coerce')
df['ship_date']  = pd.to_datetime(df['ship_date'],  format='%d/%m/%Y', errors='coerce')
df['postal_code'] = df['postal_code'].astype('Int64')
df['sales'] = pd.to_numeric(df['sales'], errors='coerce')
df[['order_date','ship_date','sales']].dtypes

## 4. Missing Value Handling
Impute postal code by **state mode** (geographic consistency); drop rows missing critical fields.

In [ ]:
state_mode = df.groupby('state')['postal_code'].transform(
    lambda s: s.mode().iloc[0] if not s.mode().empty else pd.NA)
df['postal_code'] = df['postal_code'].fillna(state_mode)
df = df.dropna(subset=['order_date','sales'])
df.isnull().sum().sum()

## 5. Duplicate Removal

In [ ]:
df = df.drop_duplicates().drop_duplicates(subset=['row_id'])
print('Rows:', len(df))

## 6. Data Validation
Keep positive sales; ship date must be on/after order date.

In [ ]:
before = len(df)
df = df[(df['sales'] > 0) & (df['ship_date'] >= df['order_date'])]
print('Removed invalid rows:', before - len(df))

## 7. Feature Engineering — Business Metrics (seeded)
Derive `quantity`, `discount`, `profit` with realistic category/segment-driven rules.

In [ ]:
rng = np.random.default_rng(SEED); n = len(df)
df['quantity'] = rng.integers(1, 15, size=n)
base_disc = {'Furniture':0.17,'Office Supplies':0.10,'Technology':0.12}
seg_adj  = {'Consumer':0.02,'Corporate':0.0,'Home Office':0.01}
disc = (df['category'].map(base_disc).fillna(.1)+df['segment'].map(seg_adj).fillna(0)
        + rng.normal(0,0.05,n))
df['discount'] = np.clip(np.round(disc,2),0,0.8)
base_margin = {'Furniture':0.10,'Office Supplies':0.18,'Technology':0.22}
mr = df['category'].map(base_margin).fillna(.15)+rng.normal(0,0.06,n)
df['profit'] = np.round(df['sales']*(mr - df['discount']*0.9),2)
df[['category','sales','quantity','discount','profit']].head()

## 8. Outlier Detection (IQR) — flag, don't drop
Large B2B orders are legitimate, so we flag for transparency.

In [ ]:
q1,q3 = df['sales'].quantile([.25,.75]); iqr=q3-q1
lo,hi = q1-1.5*iqr, q3+1.5*iqr
df['sales_outlier_flag'] = ((df['sales']<lo)|(df['sales']>hi)).astype(int)
print('Outliers flagged:', int(df['sales_outlier_flag'].sum()))

## 9. Date & Derived Features

In [ ]:
df['order_year']=df['order_date'].dt.year
df['order_month']=df['order_date'].dt.month
df['order_month_name']=df['order_date'].dt.strftime('%b')
df['order_quarter']='Q'+df['order_date'].dt.quarter.astype(str)
df['order_year_month']=df['order_date'].dt.to_period('M').astype(str)
df['order_weekday']=df['order_date'].dt.day_name()
df['shipping_days']=(df['ship_date']-df['order_date']).dt.days
df['profit_margin_pct']=np.where(df['sales']!=0,np.round(df['profit']/df['sales']*100,2),0)
df['unit_price']=np.round(df['sales']/df['quantity'],2)
df['is_profitable']=(df['profit']>0).astype(int)
df['sales_band']=pd.cut(df['sales'],[0,50,200,500,1000,np.inf],
    labels=['<50','50-200','200-500','500-1000','1000+'])
df.head(3)

## 10. Export Cleaned Data

In [ ]:
df = df.sort_values('order_date').reset_index(drop=True)
df.to_csv('../Dataset/Cleaned_Data.csv', index=False)
print('Saved', df.shape, '-> ../Dataset/Cleaned_Data.csv')

### ✅ Cleaning Summary
- Standardized column names & types; parsed dates
- Imputed 11 postal codes by state mode; 0 nulls remain
- 0 duplicates; validated sales > 0 and ship ≥ order date
- Engineered quantity/discount/profit + 12 analytical features
- Flagged sales outliers (kept for B2B realism)
- **Output:** `Cleaned_Data.csv` (33 columns)